# Imports

In [1]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter

import math
import copy
from pathlib import Path
from tqdm import trange, tqdm

from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace

# Build Transormer

## Embeddings

In [2]:
class InputEmbeddings(nn.Module):
    def __init__(self, d_model, vocab_size):
        super().__init__()
        self.d_model = d_model
        self.embeddings = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        return self.embeddings(x) * math.sqrt(self.d_model)

## Positional Encoding

In [3]:
class PositionEncoding(nn.Module):
    def __init__(self, d_model, seq_length, dropout=0.1):
        super(PositionEncoding, self).__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(seq_length, d_model)
        position = torch.arange(0, seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)  # (1, seq_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

## Layer Normalization

In [4]:
class LayerNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.alpha = nn.Parameter(torch.ones(d_model))   # shape: (d_model,)
        self.bias  = nn.Parameter(torch.zeros(d_model))  # shape: (d_model,)

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        std  = x.std(dim=-1, keepdim=True, unbiased=False)
        return self.alpha * (x - mean) / (std + self.eps) + self.bias

## Feeed Forward Layer

In [5]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear_1 = nn.Linear(d_model, d_ff)
        self.linear_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = F.relu(self.linear_1(x))
        x = self.dropout(x)
        return self.linear_2(x)

## Multi Head Attention

In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_k = d_model // num_heads
        self.dropout = nn.Dropout(dropout)

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)

        self.w_o = nn.Linear(d_model, d_model)

    @staticmethod
    def attention(query, key, value, mask, dropout):
        # query/key/value shapes: (batch, heads, seq, d_k)
        d_k = query.size(-1)

        scores = (query @ key.transpose(-2, -1)) / math.sqrt(d_k)

        if mask is not None:

            scores = scores.masked_fill(mask == 0, -1e9)

        attn = scores.softmax(dim=-1)

        if dropout is not None:
            attn = dropout(attn)

        return attn @ value, attn

    def forward(self, q, k, v, mask=None):
        # q/k/v: (batch, seq, d_model)

        batch_size = q.size(0)

        # 1) Linear projections
        query = self.w_q(q)
        key   = self.w_k(k)
        value = self.w_v(v)

        # 2) (batch, num_heads, seq, d_k)
        query = query.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        key   = key.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        value = value.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # 3) Atenção em todos os heads
        x, self.attention_scores = MultiHeadAttention.attention(query, key, value, mask, self.dropout)

        # 4) Junta heads
        x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads * self.d_k)

        # 5) Projeção final
        return self.w_o(x)

## Residual Connection

In [7]:
class ResidualConnection(nn.Module):
    def __init__(self, dropout, d_model):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.norm = LayerNorm(d_model)

    def forward(self, x, sublayer, *args, **kwargs):
        """
        sublayer: callable que recebe (normed_x, *args, **kwargs)
        """
        return x + self.dropout(sublayer(self.norm(x), *args, **kwargs))

## Encoder

In [8]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, self_attention, feed_forward, dropout=0.1):
        super().__init__()

        self.self_attention = self_attention
        self.feed_forward = feed_forward

        self.residual_connections = nn.ModuleList([
            ResidualConnection(dropout, d_model),
            ResidualConnection(dropout, d_model)
        ])

    def forward(self, x, src_mask):
        # self-attention
        x = self.residual_connections[0](
            x,
            lambda y: self.self_attention(y, y, y, src_mask)[0]  # pega só o output, ignora attn
        )

        # feed-forward
        x = self.residual_connections[1](x, self.feed_forward)
        return x

In [9]:
class Encoder(nn.Module):
    def __init__(self, layer, N, d_model):
        super().__init__()
        self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(N)])
        self.norm = LayerNorm(d_model)

    def forward(self, x, src_mask):
        for layer in self.layers:
            x = layer(x, src_mask)
        return self.norm(x)

## Decoder

In [10]:
class DecoderLayer(nn.Module):
    def __init__(self, self_attention, cross_attention, feed_forward, d_model, dropout=0.1):
        super().__init__()

        self.self_attention = self_attention
        self.cross_attention = cross_attention
        self.feed_forward = feed_forward

        # 3 residual connections
        self.residual_connections = nn.ModuleList([
            ResidualConnection(dropout, d_model),  # masked self-attention
            ResidualConnection(dropout, d_model),  # cross-attention
            ResidualConnection(dropout, d_model)   # feed-forward
        ])

    def forward(self, x, encoder_output, src_mask, tgt_mask):

        # masked self-attention
        x = self.residual_connections[0](
            x,
            lambda y: self.self_attention(y, y, y, tgt_mask)[0]
        )

        # cross-attention (decoder queries → encoder keys/values)
        x = self.residual_connections[1](
            x,
            lambda y: self.cross_attention(y, encoder_output, encoder_output, src_mask)[0]
        )

        # feed-forward
        x = self.residual_connections[2](x, self.feed_forward)

        return x

In [11]:
class Decoder(nn.Module):
  def __init__(self, layer, N, d_model):
    super(Decoder, self).__init__()
    self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(N)])
    self.norm = LayerNorm(d_model)

  def forward(self, x, encoder_output, src_mask, tgt_mask):
    for layer in self.layers:
        x = layer(x, encoder_output, src_mask, tgt_mask)
    return self.norm(x)

## Projection Layer

In [12]:
class ProjectionLayer(nn.Module):
  def __init__(self, d_model, vocab_size):
    super(ProjectionLayer, self).__init__()

    self.proj = nn.Linear(d_model, vocab_size)

  def forward(self, x):
    return torch.log_softmax(self.proj(x), dim=-1)

## Transformer

In [13]:
class Transformer(nn.Module):
  def __init__(self,
               encoder:Encoder,
               decoder:Decoder,
               src_embed:InputEmbeddings,
               target_embed:InputEmbeddings,
               src_pos:PositionEncoding,
               target_pos:PositionEncoding,
               proj_layer:ProjectionLayer
               ):

    super(Transformer, self).__init__()
    self.encode = encoder
    self.decode = decoder
    self.src_embed = src_embed
    self.src_pos = src_pos
    self.target_embed = target_embed
    self.target_pos = target_pos
    self.proj_layer = proj_layer

  def _encode(self, src, src_mask):
    src = self.src_embed(src)
    src = self.src_pos(src)
    return self.encode(src, src_mask)

  def _decode(self, encoder_output, src_mask, tgt, tgt_mask):
    tgt = self.target_embed(tgt)
    tgt = self.target_pos(tgt)
    return self.decode(tgt, encoder_output, src_mask, tgt_mask)

  def _project(self, x):
    return self.proj_layer(x)

  def forward(self, src, src_mask, tgt, tgt_mask):
    # Encoder
    output_encoder = self._encode(src, src_mask)

    # Decoder
    output_decoder = self._decode(output_encoder, src_mask, tgt, tgt_mask)

    # Projeção final
    return self._project(output_decoder)

In [14]:
def build_transformer(src_vocab_size, tgt_vocab_size,
                      src_seq_len, tgt_seq_len,
                      d_model=512, num_heads=8,
                      N=6, d_ff=2048
                      ) -> Transformer:

  src_embed = InputEmbeddings(d_model, src_vocab_size)
  tgt_embed = InputEmbeddings(d_model, tgt_vocab_size)

  src_pos = PositionEncoding(d_model, src_seq_len)
  tgt_pos = PositionEncoding(d_model, tgt_seq_len)

  # Encoder
  encoder_self_attention = MultiHeadAttention(d_model, num_heads)
  encoder_feed_forward = FeedForward(d_model, d_ff)

  encoder_layer = EncoderLayer(d_model, encoder_self_attention, encoder_feed_forward)
  encoder = Encoder(encoder_layer, N, d_model)

  # Decoder
  cross_attention = MultiHeadAttention(d_model, num_heads)
  decoder_self_attention = MultiHeadAttention(d_model, num_heads)
  decoder_feed_forward = FeedForward(d_model, d_ff)

  decoder_layer = DecoderLayer(decoder_self_attention, cross_attention, decoder_feed_forward, d_model)
  decoder = Decoder(decoder_layer, N, d_model)

  proj_layer = ProjectionLayer(d_model, tgt_vocab_size)

  transformer = Transformer(encoder=encoder,
                            decoder=decoder,
                            src_embed=src_embed,
                            target_embed=tgt_embed,
                            src_pos=src_pos,
                            target_pos=tgt_pos,
                            proj_layer=proj_layer)
  for p in transformer.parameters():
    if p.dim() > 1:
      nn.init.xavier_normal_(p)
  return transformer

# Dataset

In [15]:
class BilingualDataset(Dataset):
    def __init__(self, ds, tokenizer_src, tokenizer_tgt, src_lang, tgt_lang, seq_len):
        self.ds = ds
        self.tokenizer_src = tokenizer_src
        self.tokenizer_tgt = tokenizer_tgt
        self.src_lang = src_lang
        self.tgt_lang = tgt_lang
        self.seq_len = seq_len

        self.sos_src = tokenizer_src.token_to_id("[SOS]")
        self.eos_src = tokenizer_src.token_to_id("[EOS]")
        self.pad_src = tokenizer_src.token_to_id("[PAD]")

        self.sos_tgt = tokenizer_tgt.token_to_id("[SOS]")
        self.eos_tgt = tokenizer_tgt.token_to_id("[EOS]")
        self.pad_tgt = tokenizer_tgt.token_to_id("[PAD]")

    def causal_mask(self, size):
        mask = torch.triu(torch.ones((1, size, size)), diagonal=1).bool()
        return ~mask   # True = allowed, False = masked

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, index):
        item = self.ds[index]

        src_text = item["translation"][self.src_lang]
        tgt_text = item["translation"][self.tgt_lang]

        enc_tokens = self.tokenizer_src.encode(src_text).ids
        dec_tokens = self.tokenizer_tgt.encode(tgt_text).ids

        enc_pad = self.seq_len - len(enc_tokens) - 2
        dec_pad = self.seq_len - len(dec_tokens) - 1

        if enc_pad < 0 or dec_pad < 0:
            raise ValueError("Sentence too long")

        # Encoder input
        encoder_input = torch.tensor(
            [self.sos_src] + enc_tokens + [self.eos_src] + [self.pad_src] * enc_pad,
            dtype=torch.long
        )

        # Decoder input
        decoder_input = torch.tensor(
            [self.sos_tgt] + dec_tokens + [self.pad_tgt] * dec_pad,
            dtype=torch.long
        )

        # Labels (shifted)
        label = torch.tensor(
            dec_tokens + [self.eos_tgt] + [self.pad_tgt] * dec_pad,
            dtype=torch.long
        )

        # Masks
        enc_mask = (encoder_input != self.pad_src).unsqueeze(0).unsqueeze(0)

        dec_mask = (decoder_input != self.pad_tgt).unsqueeze(0).unsqueeze(1)
        dec_mask = dec_mask & self.causal_mask(decoder_input.size(0))

        return {
            "encoder_input": encoder_input,
            "decoder_input": decoder_input,
            "encoder_mask": enc_mask.int(),
            "decoder_mask": dec_mask.int(),
            "label": label,
            "src_text":src_text,
            "tgt_text":tgt_text
        }

# Config

In [16]:
def get_all_sentences(ds, lang):
  for item in ds:
    yield item["translation"][lang]

In [17]:
def get_or_build_tokenizer(CFG, ds, lang):
  tokenizer_path = Path(CFG["tokenizer_file"].format(lang))
  if not tokenizer_path.exists():
    tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = Whitespace()
    trainer = WordLevelTrainer(special_tokens=["[UNK]", "[PAD]", "[SOS]", "[EOS]"], min_frequency=2)
    tokenizer.train_from_iterator(get_all_sentences(ds, lang), trainer=trainer)
    tokenizer.save(str(tokenizer_path))
  else:
    tokenizer = Tokenizer.from_file(str(tokenizer_path))

  return tokenizer

In [18]:
def get_ds(config):
    src = config["lang_src"]
    tgt = config["lang_tgt"]

    ds = load_dataset("opus_books", f"{src}-{tgt}", split="train")

    tokenizer_src = get_or_build_tokenizer(config, ds, src)
    tokenizer_tgt = get_or_build_tokenizer(config, ds, tgt)

    def filter_long(example):
        src_ids = tokenizer_src.encode(example["translation"][src]).ids
        tgt_ids = tokenizer_tgt.encode(example["translation"][tgt]).ids
        return len(src_ids) <= config["seq_len"] - 2 and len(tgt_ids) <= config["seq_len"] - 1

    before = len(ds)
    ds = ds.filter(filter_long)
    after = len(ds)

    ds_train_size = int(0.9 * len(ds))
    val_ds_size = len(ds) - ds_train_size
    train, val = random_split(ds, [ds_train_size, val_ds_size])

    train_ds = BilingualDataset(train, tokenizer_src, tokenizer_tgt, src, tgt, config["seq_len"])
    val_ds   = BilingualDataset(val, tokenizer_src, tokenizer_tgt, src, tgt, config["seq_len"])

    max_len_src, max_len_tgt = 0, 0
    for item in ds.select(range(500)):
        src_len = len(tokenizer_src.encode(item["translation"][src]).ids)
        tgt_len = len(tokenizer_tgt.encode(item["translation"][tgt]).ids)
        max_len_src = max(max_len_src, src_len)
        max_len_tgt = max(max_len_tgt, tgt_len)


    train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=1, shuffle=True)

    return train_loader, val_loader, tokenizer_src, tokenizer_tgt

In [19]:
def get_weights(CFG, epoch):
    model_folder = Path(CFG["model_folder"])
    model_folder.mkdir(exist_ok=True, parents=True)

    model_file = f"{CFG['model_filename']}{epoch}.pt"
    return model_folder / model_file

In [20]:
CFG = {
    "seq_len":320,
    "batch_size": 8,
    "lang_src":"en",
    "lang_tgt":"it",
    "tokenizer_file": "tokenizer_{0}.json",
    "num_epochs":20,
    "lr":1e-4,
    "model_folder":"weights",
    "model_filename":"tmodel_",
    "preload":None,
    "experiment_name":"runs/tmodel",
}

# Training

In [21]:
def train_model(cfg):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    train_loader, val_loader, tokenizer_src, tokenizer_tgt = get_ds(cfg)

    src_vocab_size = tokenizer_src.get_vocab_size()
    tgt_vocab_size = tokenizer_tgt.get_vocab_size()

    model = build_transformer(src_vocab_size, tgt_vocab_size,
                              cfg["seq_len"], cfg["seq_len"]).to(device)

    writer = SummaryWriter(cfg["experiment_name"])
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], eps=1e-9)

    initial_epoch = 0
    global_step = 0

    if cfg["preload"]:
        model_file_name = get_weights(cfg, cfg["preload"])
        state = torch.load(model_file_name)
        initial_epoch = state["epoch"] + 1
        model.load_state_dict(state["model_state_dict"])
        optimizer.load_state_dict(state["optimizer_state_dict"])
        global_step = state["global_step"]

    loss_fn = nn.CrossEntropyLoss(
        ignore_index=tokenizer_tgt.token_to_id("[PAD]"),
        label_smoothing=0.1
    )

    for epoch in trange(cfg["num_epochs"], desc="Training", initial=initial_epoch):
        model.train()

        for batch in train_loader:
            encoder_input = batch["encoder_input"].to(device)
            decoder_input = batch["decoder_input"].to(device)
            encoder_mask = batch["encoder_mask"].to(device)
            decoder_mask = batch["decoder_mask"].to(device)
            label = batch["label"].to(device)

            # forward
            encoder_output = model._encode(encoder_input, encoder_mask)
            decoder_output = model._decode(encoder_output, encoder_mask, decoder_input, decoder_mask)
            project_output = model._project(decoder_output)

            loss = loss_fn(project_output.view(-1, tgt_vocab_size), label.view(-1))

            writer.add_scalar("Train Loss", loss.item(), global_step)

            # backward
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

            global_step += 1

        print(f"Epoch {epoch+1} Loss: {loss.item():.3f}")

        # salvar pesos
        model_file_name = get_weights(cfg, epoch)
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "global_step": global_step
        }, model_file_name)

In [ ]:
if __name__ == "__main__":
  train_model(CFG)

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Training:   0%|          | 0/20 [00:00<?, ?it/s]

Epoch 1 Loss: 5.819


Training:   5%|▌         | 1/20 [22:52<7:14:36, 1372.42s/it]